# Role Mining — Part A: User × Entitlement Matrix
**Keystone Fuel & Market (synthetic data) · Snapshot T0 · Seed 2026**

Goal: reshape access assignments into a person × entitlement matrix, measure how access is spread, and find candidate birthright access by department.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "t0"
IMG = ROOT / "reports" / "img"
IMG.mkdir(parents=True, exist_ok=True)
print("Reading data from:", DATA)

In [ ]:
hr = pd.read_csv(DATA / "hr_feed.csv")
assignments = pd.read_csv(DATA / "entitlement_assignments.csv")
catalog = pd.read_csv(DATA / "entitlement_catalog.csv")

print("HR records:      ", hr.shape)
print("Assignments:     ", assignments.shape)
print("Entitlements:    ", catalog.shape)

In [ ]:
print(hr["status"].value_counts())
print()
print(hr["department"].value_counts())

In [ ]:
joined = assignments.merge(
    hr[["upn", "employee_id", "department", "job_title", "status"]],
    on="upn", how="left", indicator=True)

print(joined["_merge"].value_counts())
no_hr = joined[joined["_merge"] == "left_only"]
print("Assignments with no HR record:", len(no_hr), "across", no_hr["upn"].nunique(), "accounts")

In [ ]:
active = joined[(joined["_merge"] == "both") & (joined["status"] == "Active")].copy()
print("Assignments kept for mining:", len(active))
print("People:", active["upn"].nunique(), "| Entitlements:", active["entitlement"].nunique())

In [ ]:
matrix = pd.crosstab(active["upn"], active["entitlement"]).clip(upper=1)
print("Matrix shape (people x entitlements):", matrix.shape)
print("Total 1s in the matrix:", int(matrix.values.sum()))
print(f"Fill rate: {matrix.values.mean():.1%}")
matrix.iloc[:5, :6]

In [ ]:
per_user = matrix.sum(axis=1)
print(per_user.describe().round(1))

In [ ]:
per_ent = matrix.sum(axis=0).sort_values(ascending=False)
print("Most-held entitlements:"); print(per_ent.head(5)); print()
print("Least-held entitlements:"); print(per_ent.tail(5))

In [ ]:
dept_of = active.drop_duplicates("upn").set_index("upn")["department"]
prevalence = matrix.groupby(dept_of.reindex(matrix.index)).mean()
prevalence.round(2).T.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(prevalence.values, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_yticks(range(len(prevalence.index)), prevalence.index)
ax.set_xticks(range(len(prevalence.columns)), prevalence.columns, rotation=90, fontsize=8)
ax.set_title("Keystone Fuel & Market (synthetic): share of each department holding each entitlement — T0")
fig.colorbar(im, ax=ax, label="Share of department")
fig.tight_layout()
fig.savefig(IMG / "m2-dept-entitlement-heatmap-t0.png", dpi=150)
plt.show()

In [ ]:
birthright_candidates = prevalence.stack().rename("share").reset_index()
birthright_candidates = birthright_candidates[birthright_candidates["share"] >= 0.90]
print("Department/entitlement pairs held by 90%+ of the department:", len(birthright_candidates))
birthright_candidates.sort_values(["department", "share"], ascending=[True, False])

## Findings (T0)
- __ active employees hold __ entitlements through __ assignments; the matrix is __% filled.
- __ assignments (__ accounts) have no HR record and were excluded from mining — they are cleanup work, not role input.
- __ department/entitlement pairs are held by 90%+ of a department (birthright candidates).
- Not every 90%+ pair should become automatic access: __ (name one, and say why).
- Next: cluster people by similar access (M2.2) — department alone is too coarse because __.